In [1]:
! pip install pyfaidx polars

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
! pip  show pyfaidx polars | grep -E "Name|Version"

Name: pyfaidx
Version: 0.9.0.4
Name: polars
Version: 1.2.1


In [5]:
import polars as pl
import pyfaidx
import numpy as np
from pathlib import Path

In [6]:
# --- paths ---
DATA_ROOT = Path.home() / "vambersky_t/Data"
BED_DIR = DATA_ROOT / "encode_beds"
REF_GENOME = DATA_ROOT / "reference" / "hg19.fa"
WINDOWS_DIR = DATA_ROOT / "extracted_windows"
TFS_TO_PROCESS = Path("tfs_to_process.txt").read_text().splitlines()

# --- parameters ---
STUDIED_SPAN = 10_000
MAX_PEAKS = 10_000
RANDOM_SEED = 42

# --- sanity checks ---
assert DATA_ROOT.exists(), f"Data root not found: {DATA_ROOT}"
assert REF_GENOME.exists(), f"Reference genome not found: {REF_GENOME}"
assert BED_DIR.exists(), f"BED directory not found: {BED_DIR}"

In [7]:
def read_bed(bed_path: Path) -> pl.DataFrame:
    """
    Read a gzipped ENCODE narrowPeak file and compute peak centres.
    Uses midpoint of start and end as the binding site proxy.
    Returns a Polars DataFrame with columns: peak_id, chr, start, end, center.
    """
    df = pl.read_csv(
        bed_path,
        separator="\t",
        has_header=False,
        comment_prefix="#",
        new_columns=["chr", "start", "end", "name", "score", "strand",
                     "signal_value", "p_value", "q_value", "peak"],
    )

    df = df.select(["chr", "start", "end"]).with_columns([
        ((pl.col("start") + pl.col("end")) // 2).alias("center"),
        pl.Series("peak_id", [f"peak_{i}" for i in range(len(df))]),
    ])

    return df

In [8]:
def subsample_peaks(df: pl.DataFrame, max_peaks: int = MAX_PEAKS, random_seed: int = RANDOM_SEED) -> pl.DataFrame:
    """
    Subsample to at most max_peaks rows with a fixed random seed.
    If fewer peaks than max_peaks are present, returns all of them.
    """
    if len(df) > max_peaks:
        print(f"Subsampling {len(df)} peaks to {max_peaks}") # maybe print this to a file?
        df = df.sample(n=max_peaks, seed=random_seed)
    else:
        print(f"Fewer than {max_peaks} peaks present ({len(df)}), keeping all")
    return df

In [9]:
def extract_sequences(df: pl.DataFrame, ref_genome: Path, studied_span: int = STUDIED_SPAN) -> pl.DataFrame:
    """
    Extract studied_span bp sequences centred on each peak centre from the reference genome.
    Non-ACGT characters are masked to N. Sequences extending beyond chromosome boundaries are dropped.
    """
    fasta = pyfaidx.Fasta(str(ref_genome))
    chrom_sizes = {k: len(v) for k, v in fasta.items()}

    sequences = []
    for row in df.iter_rows(named=True):
        chrom = row["chr"]
        center = row["center"]
        start = center - (studied_span // 2)
        end = center + (studied_span // 2)

        # drop peaks that extend beyond chromosome boundaries
        if start < 0 or end > chrom_sizes.get(chrom, 0):
            sequences.append(None)
            continue

        seq = str(fasta[chrom][start:end]).upper()
        # mask non-ACGT to N
        seq = "".join(b if b in "ACGT" else "N" for b in seq)
        sequences.append(seq)

    fasta.close()

    df = df.with_columns(
        pl.Series("sequence", sequences)
    ).filter(pl.col("sequence").is_not_null())

    return df

In [10]:
def process_bed_file(bed_path: Path) -> None:
    """
    Full pipeline for a single BED file:
    reads peaks, subsamples, extracts sequences, saves output.
    """
    # parse TF and biosample from filename
    accession, tf, biosample = bed_path.stem.replace(".bed", "").split("__")

    # output directory per TF
    output_dir = WINDOWS_DIR / tf
    output_dir.mkdir(parents=True, exist_ok=True)

    output_path = output_dir / f"{accession}__{tf}__{biosample}.full_peaks.csv.gz"

    if output_path.exists():
        print(f"Output already exists, skipping: {output_path}")
        return

    print(f"Processing {bed_path.name}")
    df = read_bed(bed_path)
    df = subsample_peaks(df)
    df = extract_sequences(df, REF_GENOME)

    df.write_csv(output_path)
    print(f"Saved {len(df)} peaks to {output_path}")

In [11]:

for tf in TFS_TO_PROCESS:
    bed_files = list(BED_DIR.glob(f"*__{tf}__*.bed.gz"))
    
    if not bed_files:
        print(f"No BED files found for {tf}")
        continue
    
    print(f"\nProcessing {tf} — {len(bed_files)} file(s) found")
    for bed_file in bed_files:
        process_bed_file(bed_file)


Processing MYC — 8 file(s) found
Output already exists, skipping: /home/jovyan/vambersky_t/Data/extracted_windows/MYC/ENCFF800JFG__MYC__HepG2.full_peaks.csv.gz
Output already exists, skipping: /home/jovyan/vambersky_t/Data/extracted_windows/MYC/ENCFF765CKK__MYC__GM12878.full_peaks.csv.gz
Output already exists, skipping: /home/jovyan/vambersky_t/Data/extracted_windows/MYC/ENCFF356PWE__MYC__A549.full_peaks.csv.gz
Output already exists, skipping: /home/jovyan/vambersky_t/Data/extracted_windows/MYC/ENCFF372RQZ__MYC__H1.full_peaks.csv.gz
Output already exists, skipping: /home/jovyan/vambersky_t/Data/extracted_windows/MYC/ENCFF043WTJ__MYC__K562.full_peaks.csv.gz
Output already exists, skipping: /home/jovyan/vambersky_t/Data/extracted_windows/MYC/ENCFF700CXD__MYC__H1.full_peaks.csv.gz
Output already exists, skipping: /home/jovyan/vambersky_t/Data/extracted_windows/MYC/ENCFF674RQX__MYC__A549.full_peaks.csv.gz
Output already exists, skipping: /home/jovyan/vambersky_t/Data/extracted_windows/MYC

InvalidOperationError: use the compression parameter to control compression, or set `check_extension` to `False` if you want to suffix an uncompressed filename with an ending intended for compression

Resolved plan until failure:

	---> FAILED HERE RESOLVING THIS_NODE <---
DF ["chr", "start", "end", "center", ...]; PROJECT */6 COLUMNS